In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/odedgolden/movielens-1m-dataset/users.dat
/kaggle/input/datasets/odedgolden/movielens-1m-dataset/ratings.dat
/kaggle/input/datasets/odedgolden/movielens-1m-dataset/README
/kaggle/input/datasets/odedgolden/movielens-1m-dataset/movies.dat


In [4]:
ratings = pd.read_csv(
    "/kaggle/input/datasets/odedgolden/movielens-1m-dataset/ratings.dat",
    sep="::", engine="python", header=None,
    names=["userId", "movieId", "rating", "timestamp"],
)

users = pd.read_csv(
    "/kaggle/input/datasets/odedgolden/movielens-1m-dataset/users.dat",
    sep="::", engine="python", header=None,
    names=["userId", "gender", "age", "occupation", "zip"],
    encoding="latin-1",
)

movies = pd.read_csv(
    "/kaggle/input/datasets/odedgolden/movielens-1m-dataset/movies.dat",
    sep="::", engine="python", header=None,
    names=["movieId", "title", "genres"],
    encoding="latin-1",
)


In [5]:
print(f"ratings: {ratings.shape}, movies: {movies.shape}, users: {users.shape}")

ratings: (1000209, 4), movies: (3883, 3), users: (6040, 5)


In [6]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   userId     1000209 non-null  int64
 1   movieId    1000209 non-null  int64
 2   rating     1000209 non-null  int64
 3   timestamp  1000209 non-null  int64
dtypes: int64(4)
memory usage: 30.5 MB


In [7]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  3883 non-null   int64 
 1   title    3883 non-null   object
 2   genres   3883 non-null   object
dtypes: int64(1), object(2)
memory usage: 91.1+ KB


In [8]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6040 entries, 0 to 6039
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   userId      6040 non-null   int64 
 1   gender      6040 non-null   object
 2   age         6040 non-null   int64 
 3   occupation  6040 non-null   int64 
 4   zip         6040 non-null   object
dtypes: int64(3), object(2)
memory usage: 236.1+ KB


In [9]:
ratings_sorted = ratings.sort_values(["userId", "timestamp"]).copy()
ratings_sorted["hour"] = pd.to_datetime(ratings_sorted["timestamp"], unit="s").dt.hour

user_seq = (
    ratings_sorted.groupby("userId")
    .agg(
        watched_item_ids=("movieId", list),
        watch_weights=("rating", lambda s: (s / 5.0).round(4).tolist()),
        hour_seq=("hour", list),
    )
    .reset_index()
)

# Bring in gender/age/occupation from users.dat; drop zip (too high-cardinality
# to be a useful categorical feature without bucketing into a region first).
user_df = (
    user_seq.merge(users.drop(columns=["zip"]), on="userId", how="left")
    .rename(columns={"userId": "user_id"})
)

# Defensive filter (every standard MovieLens release guarantees >=20
# ratings/user, but this keeps the notebook safe regardless).
min_seq_length = 3
user_df = user_df[user_df["watched_item_ids"].apply(len) >= min_seq_length].reset_index(drop=True)

print(f"User sequence + demographic data: {user_df.shape}")
user_df.head()


User sequence + demographic data: (6040, 7)


,user_id,watched_item_ids,watch_weights,hour_seq,gender,age,occupation
0,1,"[3186, 1270, 1721, 1022, 2340, 1836, 3408, 280...","[0.8, 1.0, 0.8, 1.0, 0.6, 1.0, 0.8, 1.0, 0.8, ...","[22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 2...",F,1,10
1,2,"[1198, 1210, 1217, 2717, 1293, 2943, 1225, 119...","[0.8, 0.8, 0.6, 0.6, 1.0, 0.8, 1.0, 1.0, 1.0, ...","[21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 2...",M,56,16
2,3,"[593, 2858, 3534, 1968, 1431, 1961, 1266, 1378...","[0.6, 0.8, 0.6, 0.8, 0.6, 0.8, 1.0, 1.0, 0.8, ...","[21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 2...",M,25,15
3,4,"[1210, 1097, 3468, 480, 3527, 260, 1196, 1198,...","[0.6, 0.8, 1.0, 0.8, 0.2, 1.0, 0.4, 1.0, 1.0, ...","[20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 2...",M,45,7
4,5,"[2717, 908, 919, 1250, 356, 2858, 1127, 2188, ...","[0.2, 0.8, 0.8, 1.0, 0.2, 0.8, 0.2, 0.2, 0.6, ...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...",M,25,20


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

movies = movies.copy()
movies["item_text"] = (
    movies["genres"]
    .str.replace("|", " ", regex=False)
    .str.replace("(no genres listed)", "", regex=False)
    .str.lower()
    .str.strip()
)

# Small vocabulary (~18 genre labels), so a small TF-IDF dim is plenty.
TFIDF_DIM = 50
tfidf = TfidfVectorizer(max_features=TFIDF_DIM, token_pattern=r"[a-zA-Z']+")
tfidf_matrix = tfidf.fit_transform(movies["item_text"]).toarray()
tfidf_cols = [f"tfidf_{i}" for i in range(tfidf_matrix.shape[1])]

EMOTION_LEXICON = {
    "anger": ["angry", "rage", "furious", "hate", "violent", "brutal", "war", "fight", "kill", "revenge"],
    "anticipation": ["suspense", "thrilling", "cliffhanger", "exciting", "anticipation", "eager", "sequel", "upcoming"],
    "disgust": ["disgusting", "gross", "gore", "gory", "vile", "repulsive", "nasty", "sleazy"],
    "fear": ["scary", "horror", "terrifying", "fear", "dark", "disturbing", "creepy", "haunted", "nightmare"],
    "joy": ["funny", "hilarious", "joy", "happy", "delight", "fun", "charming", "feelgood", "comedy"],
    "sadness": ["sad", "tragic", "depressing", "tearjerker", "melancholy", "loss", "grief", "heartbreaking"],
    "surprise": ["twist", "unexpected", "shocking", "surprising", "surprise"],
    "trust": ["trust", "loyal", "honest", "reliable", "faithful", "heartwarming", "wholesome"],
    "positive": ["good", "great", "excellent", "amazing", "best", "wonderful", "brilliant", "masterpiece", "love", "beautiful"],
    "negative": ["bad", "terrible", "awful", "worst", "boring", "disappointing", "waste", "poor", "dull"],
}


def emotion_scores(text):
    words = text.split()
    counts = {emo: 0 for emo in EMOTION_LEXICON}
    for w in words:
        for emo, vocab in EMOTION_LEXICON.items():
            if w in vocab:
                counts[emo] += 1
    total = sum(counts.values())
    if total == 0:
        return {f"emotion_{emo}": 0.0 for emo in EMOTION_LEXICON}
    return {f"emotion_{emo}": counts[emo] / total for emo in EMOTION_LEXICON}


emotion_df = pd.DataFrame(movies["item_text"].apply(emotion_scores).tolist())

item_df = pd.concat(
    [
        movies[["movieId", "title"]].reset_index(drop=True),
        pd.DataFrame(tfidf_matrix, columns=tfidf_cols),
        emotion_df,
    ],
    axis=1,
).rename(columns={"movieId": "unique_asset_id"})

print(f"Item feature data: {item_df.shape}")
item_df.head()


Item feature data: (3883, 32)


,unique_asset_id,title,tfidf_0,tfidf_1,tfidf_2,tfidf_3,tfidf_4,tfidf_5,tfidf_6,tfidf_7,...,emotion_anger,emotion_anticipation,emotion_disgust,emotion_fear,emotion_joy,emotion_sadness,emotion_surprise,emotion_trust,emotion_positive,emotion_negative
0,1,Toy Story (1995),0.0,0.000000,0.728901,0.591714,0.344351,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,2,Jumanji (1995),0.0,0.499814,0.000000,0.516339,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,Grumpier Old Men (1995),0.0,0.000000,0.000000,0.000000,0.573172,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,4,Waiting to Exhale (1995),0.0,0.000000,0.000000,0.000000,0.755606,0.0,0.0,0.655026,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,5,Father of the Bride Part II (1995),0.0,0.000000,0.000000,0.000000,1.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [11]:
from sklearn.preprocessing import LabelEncoder
import torch

# ===== Keep only items that appear both in ratings and in item metadata =====
all_item_ids = set(item_df["unique_asset_id"].tolist())
user_item_ids = set(i for seq in user_df["watched_item_ids"] for i in seq)
valid_item_ids = sorted(all_item_ids & user_item_ids)
print(f"Valid items (present in both ratings and metadata): {len(valid_item_ids)}")

item_encoder = LabelEncoder().fit(valid_item_ids)
user_encoder = LabelEncoder().fit(user_df["user_id"])
gender_encoder = LabelEncoder().fit(user_df["gender"])
age_encoder = LabelEncoder().fit(user_df["age"])
occupation_encoder = LabelEncoder().fit(user_df["occupation"])

user_df["user_id_enc"] = user_encoder.transform(user_df["user_id"])
user_df["gender_enc"] = gender_encoder.transform(user_df["gender"])
user_df["age_enc"] = age_encoder.transform(user_df["age"])
user_df["occupation_enc"] = occupation_encoder.transform(user_df["occupation"])

item_df = item_df[item_df["unique_asset_id"].isin(valid_item_ids)].copy()
item_df["item_id_enc"] = item_encoder.transform(item_df["unique_asset_id"])
item_df = item_df.sort_values("item_id_enc").reset_index(drop=True)

# ===== Padding-safe item embedding matrix =====
feature_cols = [c for c in item_df.columns if c not in ("unique_asset_id", "title", "item_id_enc")]
item_features = item_df[feature_cols].values.astype(np.float32)
item_matrix = torch.tensor(
    np.vstack([np.zeros((1, item_features.shape[1]), dtype=np.float32), item_features])
)  # row 0 = padding, rows 1..N = real items (item_id_enc + 1)
print(f"Item embedding matrix (with padding row): {item_matrix.shape}")

item_row_to_title = dict(zip(item_df["item_id_enc"] + 1, item_df["title"]))


Valid items (present in both ratings and metadata): 3706
Item embedding matrix (with padding row): torch.Size([3707, 30])


In [12]:
item_to_enc = {cls: i + 1 for i, cls in enumerate(item_encoder.classes_)}

MAX_SEQ_LENGTH = 20


def build_training_examples(user_df):
    """Next-item prediction examples, sliding over each user's full history,
    carrying that user's gender/age/occupation along with each example so
    the model can actually condition on them."""
    examples = []
    cols = zip(
        user_df["user_id_enc"], user_df["gender_enc"], user_df["age_enc"],
        user_df["occupation_enc"], user_df["watched_item_ids"],
    )
    for user_id_enc, gender_enc, age_enc, occupation_enc, watched in cols:
        valid_items = [item_to_enc[i] for i in watched if i in item_to_enc]
        if len(valid_items) < 2:
            continue
        for i in range(1, len(valid_items)):
            history = valid_items[:i][-MAX_SEQ_LENGTH:]
            target = valid_items[i]
            examples.append({
                "user_id": user_id_enc,
                "gender": gender_enc,
                "age": age_enc,
                "occupation": occupation_enc,
                "history": history,
                "target": target,
            })
    return examples


training_data = build_training_examples(user_df)
print(f"Training examples: {len(training_data)}")

Training examples: 994169


In [13]:
from torch.utils.data import Dataset
import torch.nn as nn


class TwoTowerDataset(Dataset):
    def __init__(self, data, max_seq_length=MAX_SEQ_LENGTH):
        self.data = data
        self.max_seq_length = max_seq_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        history = sample["history"]

        if len(history) > self.max_seq_length:
            history = history[-self.max_seq_length:]
        else:
            history = [0] * (self.max_seq_length - len(history)) + history  # 0 = padding

        return {
            "user_id": torch.tensor(sample["user_id"], dtype=torch.long),
            "gender": torch.tensor(sample["gender"], dtype=torch.long),
            "age": torch.tensor(sample["age"], dtype=torch.long),
            "occupation": torch.tensor(sample["occupation"], dtype=torch.long),
            "history": torch.tensor(history, dtype=torch.long),
            "target": torch.tensor(sample["target"], dtype=torch.long),
        }


class TwoTowerWithAttention(nn.Module):
    """Self-attention pools the item-history sequence; cross-attention fuses
    that pooled item context with the user embedding. The user embedding
    itself now also folds in gender/age/occupation via small demographic
    embeddings -- in the original code this data was collected into
    user_df but never actually passed into the network."""

    def __init__(self, num_users, num_genders, num_ages, num_occupations, item_emb_matrix,
                 embed_dim=256, num_heads=4, demo_dim=8, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim

        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.gender_emb = nn.Embedding(num_genders, demo_dim)
        self.age_emb = nn.Embedding(num_ages, demo_dim)
        self.occupation_emb = nn.Embedding(num_occupations, demo_dim)
        self.demo_proj = nn.Linear(embed_dim + demo_dim * 3, embed_dim)

        # padding_idx=0 keeps the reserved padding row's gradient at zero.
        self.item_emb = nn.Embedding.from_pretrained(item_emb_matrix, freeze=False, padding_idx=0)

        if item_emb_matrix.shape[1] != embed_dim:
            self.item_proj = nn.Linear(item_emb_matrix.shape[1], embed_dim)
        else:
            self.item_proj = nn.Identity()

        self.self_attention = nn.MultiheadAttention(
            embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.cross_attention = nn.MultiheadAttention(
            embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )

        self.user_proj = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim),
        )

        self.layer_norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, user_id, gender, age, occupation, item_history, attention_mask=None):
        id_emb = self.user_emb(user_id)  # (B, D)
        demo = torch.cat(
            [id_emb, self.gender_emb(gender), self.age_emb(age), self.occupation_emb(occupation)], dim=-1
        )
        user_emb = self.demo_proj(demo)  # (B, D) -- id + demographics fused

        item_emb = self.item_emb(item_history)  # (B, L, item_dim)
        item_emb = self.item_proj(item_emb)  # (B, L, D)

        if attention_mask is None:
            attention_mask = (item_history != 0).float()  # (B, L)

        item_attended, _ = self.self_attention(item_emb, item_emb, item_emb)
        item_attended = self.layer_norm(item_attended + item_emb)

        mask_expanded = attention_mask.unsqueeze(-1).expand_as(item_attended)
        item_sum = (item_attended * mask_expanded).sum(dim=1)
        item_count = attention_mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        item_agg = item_sum / item_count  # (B, D)

        user_expanded = user_emb.unsqueeze(1)  # (B, 1, D)
        item_agg_expanded = item_agg.unsqueeze(1)  # (B, 1, D)

        user_attended, _ = self.cross_attention(user_expanded, item_agg_expanded, item_agg_expanded)
        user_attended = user_attended.squeeze(1)  # (B, D)

        user_final = self.user_proj(torch.cat([user_emb, user_attended], dim=-1))
        user_final = self.layer_norm(user_final)
        return user_final

    def get_item_embedding(self, item_ids):
        item_emb = self.item_emb(item_ids)
        return self.item_proj(item_emb)


In [14]:
import torch.nn.functional as F
from tqdm import tqdm


def train_epoch(model, train_loader, optimizer, device):
    model.train()
    total_loss = 0
    num_batches = 0

    progress_bar = tqdm(train_loader, desc="Training")
    for batch in progress_bar:
        user_id = batch["user_id"].to(device)
        gender = batch["gender"].to(device)
        age = batch["age"].to(device)
        occupation = batch["occupation"].to(device)
        history = batch["history"].to(device)
        target = batch["target"].to(device)

        attention_mask = (history != 0).float()
        user_repr = model(user_id, gender, age, occupation, history, attention_mask)

        batch_size = user_repr.size(0)
        all_targets = target.unsqueeze(0).expand(batch_size, -1)  # (B, B)
        all_target_emb = model.get_item_embedding(all_targets.reshape(-1)).reshape(batch_size, batch_size, -1)

        # In-batch negative sampling: every other item in the batch is a
        # negative for this user; the positive sits on the diagonal.
        scores = torch.bmm(user_repr.unsqueeze(1), all_target_emb.transpose(1, 2)).squeeze(1)  # (B, B)
        labels = torch.arange(batch_size, device=device)
        loss = F.cross_entropy(scores, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1
        progress_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

    return total_loss / num_batches


def evaluate_model(model, val_loader, device, k=10):
    model.eval()
    total_hits = 0
    total_samples = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):
            user_id = batch["user_id"].to(device)
            gender = batch["gender"].to(device)
            age = batch["age"].to(device)
            occupation = batch["occupation"].to(device)
            history = batch["history"].to(device)
            target = batch["target"].to(device)

            attention_mask = (history != 0).float()
            user_repr = model(user_id, gender, age, occupation, history, attention_mask)

            batch_size = user_repr.size(0)
            all_targets = target.unsqueeze(0).expand(batch_size, -1)
            all_target_emb = model.get_item_embedding(all_targets.reshape(-1)).reshape(batch_size, batch_size, -1)
            scores = torch.bmm(user_repr.unsqueeze(1), all_target_emb.transpose(1, 2)).squeeze(1)

            _, top_k_indices = torch.topk(scores, k=min(k, batch_size), dim=1)
            for i in range(batch_size):
                if i in top_k_indices[i]:
                    total_hits += 1
                total_samples += 1

    return total_hits / total_samples if total_samples > 0 else 0

In [15]:
DATA_DIR = globals().get("DATA_DIR", "/kaggle/working/data")
ARTIFACT_DIR = globals().get("ARTIFACT_DIR", "/kaggle/working/artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

In [ ]:
import gc
import pickle
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import torch.optim as optim

# Hyperparameters — scaled down from the original (256-dim, 424K users) to
# suit MovieLens ml-1m (~6K users, ~3.7K items).
EMBED_DIM = 128
NUM_HEADS = 4
DEMO_DIM = 8
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
NUM_EPOCHS = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_data, val_data = train_test_split(training_data, test_size=0.2, random_state=42)

train_loader = DataLoader(TwoTowerDataset(train_data), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(TwoTowerDataset(val_data), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

model = TwoTowerWithAttention(
    num_users=len(user_encoder.classes_),
    num_genders=len(gender_encoder.classes_),
    num_ages=len(age_encoder.classes_),
    num_occupations=len(occupation_encoder.classes_),
    item_emb_matrix=item_matrix,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    demo_dim=DEMO_DIM,
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=1, factor=0.5)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

best_hit_rate = 0
model_path = os.path.join(ARTIFACT_DIR, "best_two_tower_model.pth")
item_matrix_path = os.path.join(ARTIFACT_DIR, "item_matrix.pth")
encoders_path = os.path.join(ARTIFACT_DIR, "encoders.pkl")

for epoch in range(NUM_EPOCHS):
    print(f"\n=== Epoch {epoch + 1}/{NUM_EPOCHS} ===")

    train_loss = train_epoch(model, train_loader, optimizer, device)
    print(f"Train Loss: {train_loss:.4f}")

    hit_rate = evaluate_model(model, val_loader, device, k=10)
    print(f"Hit Rate@10: {hit_rate:.4f}")

    scheduler.step(hit_rate)

    if hit_rate > best_hit_rate:
        best_hit_rate = hit_rate
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "hit_rate": hit_rate,
                "embed_dim": EMBED_DIM,
                "num_heads": NUM_HEADS,
                "demo_dim": DEMO_DIM,
                "num_genders": len(gender_encoder.classes_),
                "num_ages": len(age_encoder.classes_),
                "num_occupations": len(occupation_encoder.classes_),
            },
            model_path,
        )
        print(f"New best model saved! Hit Rate: {hit_rate:.4f}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

torch.save(item_matrix, item_matrix_path)
with open(encoders_path, "wb") as f:
    pickle.dump(
        {
            "user_encoder": user_encoder,
            "item_encoder": item_encoder,
            "gender_encoder": gender_encoder,
            "age_encoder": age_encoder,
            "occupation_encoder": occupation_encoder,
        },
        f,
    )

print(f"\nTraining complete! Best Hit Rate@10: {best_hit_rate:.4f}")
print("Saved artifacts:")
print(f"- {model_path}")
print(f"- {item_matrix_path}")
print(f"- {encoders_path}")

Using device: cuda
Train batches: 6214, Val batches: 1554
Model parameters: 1,089,882

=== Epoch 1/10 ===


Training:  42%|████▏     | 2633/6214 [00:34<01:57, 30.57it/s, Loss=3.6409]

In [35]:
from sklearn.metrics.pairwise import cosine_similarity

# ===== Inference: load the saved artifacts and generate recommendations =====
OCCUPATION_LABELS = {
    0: "other", 1: "academic/educator", 2: "artist", 3: "clerical/admin",
    4: "college/grad student", 5: "customer service", 6: "doctor/health care",
    7: "executive/managerial", 8: "farmer", 9: "homemaker", 10: "K-12 student",
    11: "lawyer", 12: "programmer", 13: "retired", 14: "sales/marketing",
    15: "scientist", 16: "self-employed", 17: "technician/engineer",
    18: "tradesman/craftsman", 19: "unemployed", 20: "writer",
}
GENDER_LABELS = {"M": "male", "F": "female"}


def load_recommender(model_path=model_path, item_matrix_path=item_matrix_path, encoders_path=encoders_path):
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    with open(encoders_path, "rb") as f:
        encoders = pickle.load(f)
    saved_item_matrix = torch.load(item_matrix_path, map_location=device)

    loaded_model = TwoTowerWithAttention(
        num_users=len(encoders["user_encoder"].classes_),
        num_genders=checkpoint["num_genders"],
        num_ages=checkpoint["num_ages"],
        num_occupations=checkpoint["num_occupations"],
        item_emb_matrix=saved_item_matrix,
        embed_dim=checkpoint.get("embed_dim", EMBED_DIM),
        num_heads=checkpoint.get("num_heads", NUM_HEADS),
        demo_dim=checkpoint.get("demo_dim", DEMO_DIM),
    ).to(device)
    loaded_model.load_state_dict(checkpoint["model_state_dict"])
    loaded_model.eval()

    return loaded_model, encoders


def extract_user_vector(loaded_model, row, item_to_enc, max_seq_length=MAX_SEQ_LENGTH):
    history = [item_to_enc[i] for i in row["watched_item_ids"] if i in item_to_enc]
    history = history[-max_seq_length:]
    history = [0] * (max_seq_length - len(history)) + history

    user_tensor = torch.tensor([row["user_id_enc"]], dtype=torch.long, device=device)
    gender_tensor = torch.tensor([row["gender_enc"]], dtype=torch.long, device=device)
    age_tensor = torch.tensor([row["age_enc"]], dtype=torch.long, device=device)
    occupation_tensor = torch.tensor([row["occupation_enc"]], dtype=torch.long, device=device)
    history_tensor = torch.tensor([history], dtype=torch.long, device=device)
    mask_tensor = (history_tensor != 0).float()

    with torch.no_grad():
        user_repr = loaded_model(
            user_tensor, gender_tensor, age_tensor, occupation_tensor, history_tensor, mask_tensor
        )
    return user_repr.cpu().numpy()[0]


def recommend_top_k(user_vec, all_item_emb, item_encoder, k=10):
    sims = cosine_similarity([user_vec], all_item_emb)[0]
    top_k_idx = sims.argsort()[::-1][:k]
    return [
        {
            "movieId": int(item_encoder.classes_[idx]),
            "title": item_row_to_title.get(int(idx) + 1, "<unknown>"),
            "score": float(sims[idx]),
        }
        for idx in top_k_idx
    ]


rec_model, rec_encoders = load_recommender()
rec_item_encoder = rec_encoders["item_encoder"]

with torch.no_grad():
    all_item_rows = torch.arange(1, len(rec_item_encoder.classes_) + 1, device=device)
    all_item_emb = rec_model.get_item_embedding(all_item_rows).cpu().numpy()

for user_id in user_df["user_id"].sample(3, random_state=0):
    row = user_df[user_df["user_id"] == user_id].iloc[0]
    user_vec = extract_user_vector(rec_model, row, item_to_enc)
    recs = recommend_top_k(user_vec, all_item_emb, rec_item_encoder, k=10)

    gender_label = GENDER_LABELS.get(row["gender"], row["gender"])
    occupation_label = OCCUPATION_LABELS.get(int(row["occupation"]), str(row["occupation"]))
    print(
        f"User {user_id} — {gender_label}, age {row['age']}, {occupation_label} "
        f"(watched {len(row['watched_item_ids'])} movies) — top 10 recommendations:"
    )
    for rank, r in enumerate(recs, start=1):
        print(f"  {rank}. {r['title']}  (score={r['score']:.3f})")
    print()


User 207 — male, age 25, programmer (watched 23 movies) — top 10 recommendations:
  1. Any Given Sunday (1999)  (score=0.564)
  2. Abbott and Costello Meet Frankenstein (1948)  (score=0.545)
  3. Mission to Mars (2000)  (score=0.537)
  4. Snow Day (2000)  (score=0.530)
  5. Skulls, The (2000)  (score=0.526)
  6. Waking the Dead (1999)  (score=0.524)
  7. Hellbound: Hellraiser II (1988)  (score=0.524)
  8. Hideaway (1995)  (score=0.523)
  9. Voyage to the Bottom of the Sea (1961)  (score=0.523)
  10. Supernova (2000)  (score=0.522)

User 4946 — female, age 35, academic/educator (watched 257 movies) — top 10 recommendations:
  1. Lawrence of Arabia (1962)  (score=0.625)
  2. Stalag 17 (1953)  (score=0.570)
  3. Battleship Potemkin, The (Bronenosets Potyomkin) (1925)  (score=0.566)
  4. Paths of Glory (1957)  (score=0.563)
  5. Searchers, The (1956)  (score=0.552)
  6. Shane (1953)  (score=0.545)
  7. Richard III (1995)  (score=0.528)
  8. Great Escape, The (1963)  (score=0.527)
  9. Gone

In [36]:
# EVAL SETUP — leave-one-out protocol, no sliding-window leakage
import numpy as np
import pandas as pd
import torch
from collections import Counter

model.eval()
N_ITEMS = len(item_encoder.classes_)          # 3706 real items; rows 1..N_ITEMS (0 = pad)

# Rebuild each user's full encoded history (rows match item_to_enc / the padded matrix)
user_rows = []
for _, r in user_df.iterrows():
    seq = [item_to_enc[i] for i in r["watched_item_ids"] if i in item_to_enc]
    if len(seq) < 3:
        continue
    user_rows.append({
        "user_id": int(r["user_id_enc"]),
        "gender": int(r["gender_enc"]),
        "age": int(r["age_enc"]),
        "occupation": int(r["occupation_enc"]),
        "full_seq": seq,                       # last element = LOO test target
    })
print(f"Eval users: {len(user_rows)}")

# ---- Catalog item embeddings -------------------------------------------------
with torch.no_grad():
    rows_1n = torch.arange(1, N_ITEMS + 1, device=device)
    warm_cat = model.get_item_embedding(rows_1n)                       # trained embeddings
    cold_cat = model.item_proj(item_matrix.to(device))[1:]            # CONTENT-ONLY (cold-start)
print(f"warm_cat {tuple(warm_cat.shape)} | cold_cat {tuple(cold_cat.shape)}")

# ---- User-vector builder (batched, LOO history, optional truncation) --------
@torch.no_grad()
def compute_user_vectors(rows, hist_len=MAX_SEQ_LENGTH, max_len=MAX_SEQ_LENGTH, bs=4096):
    uid = torch.tensor([r["user_id"]     for r in rows], dtype=torch.long, device=device)
    g   = torch.tensor([r["gender"]      for r in rows], dtype=torch.long, device=device)
    a   = torch.tensor([r["age"]         for r in rows], dtype=torch.long, device=device)
    o   = torch.tensor([r["occupation"]  for r in rows], dtype=torch.long, device=device)
    H = []
    for r in rows:
        h = r["full_seq"][:-1]                        # drop the target
        h = h[-hist_len:] if hist_len > 0 else []     # hist_len=0 -> cold user
        H.append([0] * (max_len - len(h)) + h)
    H = torch.tensor(H, dtype=torch.long, device=device)
    mask = (H != 0).float()
    out = []
    for i in range(0, len(rows), bs):
        sl = slice(i, i + bs)
        out.append(model(uid[sl], g[sl], a[sl], o[sl], H[sl], mask[sl]))
    return torch.cat(out, 0)

# ---- Full-catalog ranking metrics -----------------------------------------
@torch.no_grad()
def ranking_metrics(user_vec, cat_emb, rows, ks=(1, 5, 10, 20, 50)):
    scores = user_vec @ cat_emb.T                     # (U, N)  col j -> item row j+1
    tcol = torch.tensor([r["full_seq"][-1] - 1 for r in rows], device=device)
    tgt_score = scores[torch.arange(len(rows)), tcol].clone()
    for u, r in enumerate(rows):                      # mask items already seen
        idx = torch.tensor([x - 1 for x in set(r["full_seq"][:-1])], device=device)
        scores[u, idx] = -1e9
    scores[torch.arange(len(rows)), tcol] = tgt_score
    rank = (scores > tgt_score.unsqueeze(1)).sum(1) + 1          # 1-based rank of target
    m = {}
    for k in ks:
        hit = (rank <= k).float()
        m[f"HR@{k}"]   = hit.mean().item()
        m[f"NDCG@{k}"] = (hit / torch.log2(rank.clamp(max=k).float() + 1)).mean().item()
    m["MRR"]        = (1.0 / rank.float()).mean().item()
    m["MedianRank"] = rank.median().item()
    m["Accuracy(HR@1)"] = m["HR@1"]
    return m, rank.cpu().numpy()


Eval users: 6040
warm_cat (3706, 128) | cold_cat (3706, 128)


We perform a leave-one-out full-catalog ranking eval against baseline. The popularity baseline just recommends whatever globally most-watched, with no personalization. This demographic tells us if the model has learnt anything or just returns most popular items.

In [37]:
# RANKING METRICS (primary) — model vs popularity baseline, full 3.7K catalog
uv_full = compute_user_vectors(user_rows, hist_len=MAX_SEQ_LENGTH)
model_m, model_rank = ranking_metrics(uv_full, warm_cat, user_rows)

# Popularity baseline: rank every item by how often it is a training target
tc = Counter(ex["target"] for ex in training_data)
pop = torch.tensor([tc.get(j + 1, 0) for j in range(N_ITEMS)], dtype=torch.float, device=device)
pop_scores = pop.unsqueeze(0).repeat(len(user_rows), 1)
for u, r in enumerate(user_rows):
    idx = torch.tensor([x - 1 for x in set(r["full_seq"][:-1])], device=device)
    pop_scores[u, idx] = -1e9
tcol = torch.tensor([r["full_seq"][-1] - 1 for r in user_rows], device=device)
pr_rank = (pop_scores > pop_scores[torch.arange(len(user_rows)), tcol].unsqueeze(1)).sum(1) + 1
pop_m = {f"HR@{k}": (pr_rank <= k).float().mean().item() for k in (1, 5, 10, 20, 50)}
pop_m["MRR"] = (1.0 / pr_rank.float()).mean().item()

print(pd.DataFrame({"two_tower": model_m, "popularity": pop_m}).round(4))

                two_tower  popularity
HR@1               0.0192      0.0055
NDCG@1             0.0192         NaN
HR@5               0.0851      0.0209
NDCG@5             0.0518         NaN
HR@10              0.1435      0.0376
NDCG@10            0.0707         NaN
HR@20              0.2320      0.0687
NDCG@20            0.0928         NaN
HR@50              0.3811      0.1411
NDCG@50            0.1224         NaN
MRR                0.0628      0.0201
MedianRank        97.0000         NaN
Accuracy(HR@1)     0.0192         NaN


In [38]:
# CLASSIFICATION FRAMING — 1 true item vs 100 sampled negatives per user
#   AUC (ROC + PR), F1, precision/recall, balanced acc, raw acc
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_curve, f1_score, accuracy_score,
                             balanced_accuracy_score, precision_score, recall_score)

N_NEG = 100
rng = np.random.default_rng(42)
uv_full = compute_user_vectors(user_rows, hist_len=MAX_SEQ_LENGTH)

score_chunks, label_chunks, per_user_auc = [], [], []
for u, r in enumerate(user_rows):
    pos = r["full_seq"][-1] - 1
    seen = {x - 1 for x in r["full_seq"]}
    negs = set()
    while len(negs) < N_NEG:
        c = int(rng.integers(0, N_ITEMS))
        if c not in seen:
            negs.add(c)
    cand = torch.tensor([pos] + list(negs), device=device)
    s = (uv_full[u:u + 1] @ warm_cat[cand].T).squeeze(0).cpu().numpy()
    lab = np.zeros(N_NEG + 1); lab[0] = 1.0
    score_chunks.append(s); label_chunks.append(lab)
    per_user_auc.append(roc_auc_score(lab, s))

S = np.concatenate(score_chunks); L = np.concatenate(label_chunks)
print(f"ROC-AUC (pooled)     : {roc_auc_score(L, S):.4f}")
print(f"ROC-AUC (mean/user)  : {np.mean(per_user_auc):.4f}")
print(f"PR-AUC  (avg prec.)  : {average_precision_score(L, S):.4f}   (positive rate = {L.mean():.3f})")

# Threshold: calibrate best-F1 on half the users, evaluate on the other half
even = [i for i in range(len(user_rows)) if i % 2 == 0]
odd  = [i for i in range(len(user_rows)) if i % 2 == 1]
cal_s = np.concatenate([score_chunks[i] for i in even]); cal_l = np.concatenate([label_chunks[i] for i in even])
ev_s  = np.concatenate([score_chunks[i] for i in odd ]); ev_l  = np.concatenate([label_chunks[i] for i in odd ])
p, rc, thr = precision_recall_curve(cal_l, cal_s)
f1_curve = 2 * p * rc / (p + rc + 1e-9)
best_t = thr[np.argmax(f1_curve[:-1])]
pred = (ev_s >= best_t).astype(int)

print(f"\n threshold (best-F1 on calib split) = {best_t:.3f}")
print(f"F1 (positive class)  : {f1_score(ev_l, pred):.4f}")
print(f"Precision / Recall   : {precision_score(ev_l, pred):.4f} / {recall_score(ev_l, pred):.4f}")
print(f"Balanced accuracy    : {balanced_accuracy_score(ev_l, pred):.4f}")
print(f"Raw accuracy         : {accuracy_score(ev_l, pred):.4f}   <- inflated by 100:1 imbalance, cite balanced acc instead")

ROC-AUC (pooled)     : 0.8813
ROC-AUC (mean/user)  : 0.8903
PR-AUC  (avg prec.)  : 0.1682   (positive rate = 0.010)

 threshold (best-F1 on calib split) = 5.945
F1 (positive class)  : 0.2592
Precision / Recall   : 0.2355 / 0.2881
Balanced accuracy    : 0.6394
Raw accuracy         : 0.9837   <- inflated by 100:1 imbalance, cite balanced acc instead


In [39]:
# COLD-START (ITEMS) — can the model rank items it barely/never trained on using only their TF-IDF
counts_arr = np.array([tc.get(j + 1, 0) for j in range(N_ITEMS)])
COLD_THR = 5
cold_mask = torch.tensor(counts_arr <= COLD_THR, device=device)
print(f"Cold items (<= {COLD_THR} train occurrences): {int(cold_mask.sum())} / {N_ITEMS}")

# Realistic serving catalog: cold items use CONTENT-ONLY embedding, warm items use trained
hybrid_cat = warm_cat.clone()
hybrid_cat[cold_mask] = cold_cat[cold_mask]

cold_users = [r for r in user_rows if counts_arr[r["full_seq"][-1] - 1] <= COLD_THR]
warm_users = [r for r in user_rows if counts_arr[r["full_seq"][-1] - 1] >  COLD_THR]
print(f"Users whose LOO target is a cold item: {len(cold_users)}")

def _eval(rows, cat, tag):
    if not rows:
        print(f"  {tag:<28} (no users)"); return
    uv = compute_user_vectors(rows, hist_len=MAX_SEQ_LENGTH)
    m, _ = ranking_metrics(uv, cat, rows)
    print(f"  {tag:<28} HR@10={m['HR@10']:.4f}  NDCG@10={m['NDCG@10']:.4f}  MRR={m['MRR']:.4f}  medRank={m['MedianRank']:.0f}")

print("\nCold-item targets:")
_eval(cold_users, cold_cat,   "content-only catalog")     # what a brand-new item gets
_eval(cold_users, hybrid_cat, "hybrid (serving-realistic)")
_eval(cold_users, warm_cat,   "trained catalog (ref)")
print("Warm-item targets (reference):")
_eval(warm_users[:2000], warm_cat, "trained catalog")


Cold items (<= 5 train occurrences): 331 / 3706
Users whose LOO target is a cold item: 7

Cold-item targets:
  content-only catalog         HR@10=0.1429  NDCG@10=0.1429  MRR=0.1455  medRank=647
  hybrid (serving-realistic)   HR@10=0.1429  NDCG@10=0.0714  MRR=0.0482  medRank=1151
  trained catalog (ref)        HR@10=0.2857  NDCG@10=0.1981  MRR=0.1845  medRank=26
Warm-item targets (reference):
  trained catalog              HR@10=0.1595  NDCG@10=0.0778  MRR=0.0675  medRank=82


In [40]:
# COLD-START (USERS) — degradation as history shrinks; unknown user-ID case
rows_r = []
for h in [0, 1, 2, 3, 5, 10, 20]:
    uv = compute_user_vectors(user_rows, hist_len=h)
    m, _ = ranking_metrics(uv, warm_cat, user_rows)
    rows_r.append({"history_len": h, "HR@10": m["HR@10"], "NDCG@10": m["NDCG@10"], "MRR": m["MRR"]})
print("Known user ID + demographics, varying history:")
print(pd.DataFrame(rows_r).round(4).to_string(index=False))

# Simulate an unseen user ID: replace every ID embedding with the mean (no personalization)
with torch.no_grad():
    orig = model.user_emb.weight.data.clone()
    model.user_emb.weight.data[:] = orig.mean(0, keepdim=True)

rows_c = []
for h in [0, 5, 20]:
    uv = compute_user_vectors(user_rows, hist_len=h)
    m, _ = ranking_metrics(uv, warm_cat, user_rows)
    rows_c.append({"history_len": h, "HR@10": m["HR@10"], "NDCG@10": m["NDCG@10"], "MRR": m["MRR"]})

with torch.no_grad():
    model.user_emb.weight.data.copy_(orig)           # restore

print("\nUnknown user ID (mean embedding) + demographics, varying history:")
print(pd.DataFrame(rows_c).round(4).to_string(index=False))
print("\nCompare row-by-row: gap = value of the learned per-user ID embedding.")
print("history_len=0 + unknown ID  ==  pure demographic cold-start (new user, no clicks).")

Known user ID + demographics, varying history:
 history_len  HR@10  NDCG@10    MRR
           0 0.0325   0.0148 0.0163
           1 0.0046   0.0018 0.0035
           2 0.0129   0.0057 0.0072
           3 0.0200   0.0088 0.0098
           5 0.0382   0.0170 0.0168
          10 0.0733   0.0329 0.0308
          20 0.1435   0.0707 0.0628

Unknown user ID (mean embedding) + demographics, varying history:
 history_len  HR@10  NDCG@10    MRR
           0 0.0065   0.0024 0.0044
           5 0.0252   0.0109 0.0116
          20 0.1184   0.0572 0.0514

Compare row-by-row: gap = value of the learned per-user ID embedding.
history_len=0 + unknown ID  ==  pure demographic cold-start (new user, no clicks).


In [ ]:
# loading model later
import os
import pickle
import torch
import torch.nn as nn


class TwoTowerWithAttention(nn.Module):
    """Must match the architecture used at training time exactly."""

    def __init__(self, num_users, num_genders, num_ages, num_occupations, item_emb_matrix,
                 embed_dim=256, num_heads=4, demo_dim=8, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim

        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.gender_emb = nn.Embedding(num_genders, demo_dim)
        self.age_emb = nn.Embedding(num_ages, demo_dim)
        self.occupation_emb = nn.Embedding(num_occupations, demo_dim)
        self.demo_proj = nn.Linear(embed_dim + demo_dim * 3, embed_dim)

        self.item_emb = nn.Embedding.from_pretrained(item_emb_matrix, freeze=False, padding_idx=0)

        if item_emb_matrix.shape[1] != embed_dim:
            self.item_proj = nn.Linear(item_emb_matrix.shape[1], embed_dim)
        else:
            self.item_proj = nn.Identity()

        self.self_attention = nn.MultiheadAttention(
            embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.cross_attention = nn.MultiheadAttention(
            embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )

        self.user_proj = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim),
        )

        self.layer_norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, user_id, gender, age, occupation, item_history, attention_mask=None):
        id_emb = self.user_emb(user_id)
        demo = torch.cat(
            [id_emb, self.gender_emb(gender), self.age_emb(age), self.occupation_emb(occupation)], dim=-1
        )
        user_emb = self.demo_proj(demo)

        item_emb = self.item_emb(item_history)
        item_emb = self.item_proj(item_emb)

        if attention_mask is None:
            attention_mask = (item_history != 0).float()

        item_attended, _ = self.self_attention(item_emb, item_emb, item_emb)
        item_attended = self.layer_norm(item_attended + item_emb)

        mask_expanded = attention_mask.unsqueeze(-1).expand_as(item_attended)
        item_sum = (item_attended * mask_expanded).sum(dim=1)
        item_count = attention_mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        item_agg = item_sum / item_count

        user_expanded = user_emb.unsqueeze(1)
        item_agg_expanded = item_agg.unsqueeze(1)

        user_attended, _ = self.cross_attention(user_expanded, item_agg_expanded, item_agg_expanded)
        user_attended = user_attended.squeeze(1)

        user_final = self.user_proj(torch.cat([user_emb, user_attended], dim=-1))
        user_final = self.layer_norm(user_final)
        return user_final

    def get_item_embedding(self, item_ids):
        item_emb = self.item_emb(item_ids)
        return self.item_proj(item_emb)


ARTIFACT_DIR = "/kaggle/working/artifacts"
model_path = os.path.join(ARTIFACT_DIR, "best_two_tower_model.pth")
item_matrix_path = os.path.join(ARTIFACT_DIR, "item_matrix.pth")
encoders_path = os.path.join(ARTIFACT_DIR, "encoders.pkl")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(model_path, map_location=device, weights_only=False)
with open(encoders_path, "rb") as f:
    encoders = pickle.load(f)
loaded_item_matrix = torch.load(item_matrix_path, map_location=device)

loaded_model = TwoTowerWithAttention(
    num_users=len(encoders["user_encoder"].classes_),
    num_genders=checkpoint["num_genders"],
    num_ages=checkpoint["num_ages"],
    num_occupations=checkpoint["num_occupations"],
    item_emb_matrix=loaded_item_matrix,
    embed_dim=checkpoint.get("embed_dim", 128),
    num_heads=checkpoint.get("num_heads", 4),
    demo_dim=checkpoint.get("demo_dim", 8),
).to(device)
loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_model.eval()

print(f"Loaded model from epoch {checkpoint['epoch'] + 1}, Hit Rate@10 = {checkpoint['hit_rate']:.4f}")